# Limpieza — Fase 3: Teléfono, duplicados parciales y ensamblaje final

In [1]:
import os
import re
import sys

import pandas as pd
from rapidfuzz import fuzz

sys.path.append(os.path.join("..", "scripts"))
from limpieza_utils import CARPETA_DATA, CARPETA_INTERIM, es_vacio, RegistroTransformaciones

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

RUTA_ENTRADA = os.path.join(CARPETA_INTERIM, "fase2_texto_libre.csv")
RUTA_CATALOGO = os.path.join(CARPETA_DATA, "catalogo_departamentos_municipios.csv")
RUTA_CRUDO = os.path.join(CARPETA_DATA, "establecimientos_diversificado_crudo.csv")

df = pd.read_csv(RUTA_ENTRADA, dtype=str, keep_default_na=False, encoding="utf-8-sig")
df = df.replace("", pd.NA)  # el redondeo por CSV convierte los NA de las fases 1-2 en cadena vacía; se restauran
catalogo = pd.read_csv(RUTA_CATALOGO, dtype=str, keep_default_na=False, encoding="utf-8-sig")

log = RegistroTransformaciones("Fase 3 - Teléfono, duplicados y ensamblaje")

print(f"Filas: {len(df)}  |  Columnas: {df.shape[1]}")


Filas: 11867  |  Columnas: 18


## TELEFONO

Problemas: 7.97% vacío, 9 celdas con letras (labels como "FAX", "Y", "AL", no texto ilegible), 184 celdas con más de un número (separadas por `-`, `/`, `,`), 49 con menos de 8 dígitos.

**Regla:** se extraen las secuencias de dígitos de la celda (ignorando letras/labels). El **primer** número encontrado va a `TELEFONO`; el **segundo** (si existe) a una columna derivada `TELEFONO_2`, en vez de descartarlo. Un guion entre dos fragmentos cortos (≤4 dígitos cada uno, ej. `331-2247`) se interpreta como el separador *interno* de un solo número (grupo 3+4); en cualquier otro caso el guion/`/`/`,` separa dos números distintos. Solo se acepta como válido un número de **exactamente 8 dígitos**; todo lo demás queda `NA` (no se inventan dígitos).


In [2]:
def candidatos_telefono(valor):
    """Devuelve la lista ordenada de numeros candidatos encontrados en la celda."""
    if pd.isna(valor) or str(valor).strip() == "":
        return []
    grupos = re.split(r"[,/]", str(valor))
    candidatos = []
    for g in grupos:
        partes = [p for p in re.split(r"-", g) if re.search(r"\d", p)]
        if len(partes) == 2 and all(len(re.sub(r"\D", "", p)) <= 4 for p in partes):
            candidatos.append(re.sub(r"\D", "", g))
        else:
            candidatos.extend(re.findall(r"\d+", g))
    return candidatos


def formatear_si_valido(numero):
    if numero is None or len(numero) != 8:
        return pd.NA
    return f"{numero[:4]}-{numero[4:]}"


antes_vacio_tel = df["TELEFONO"].map(es_vacio).sum()
antes_letras_tel = df["TELEFONO"].str.contains(r"[A-Za-z]", regex=True, na=False).sum()
antes_multiples_tel = df["TELEFONO"].str.contains(r"[-/,]", regex=True, na=False).sum()

candidatos = df["TELEFONO"].map(candidatos_telefono)
telefono_1_crudo = candidatos.map(lambda c: c[0] if len(c) >= 1 else None)
telefono_2_crudo = candidatos.map(lambda c: c[1] if len(c) >= 2 else None)

df["TELEFONO_2"] = telefono_2_crudo.map(formatear_si_valido)
df["TELEFONO"] = telefono_1_crudo.map(formatear_si_valido)

despues_valido_tel = df["TELEFONO"].notna().sum()
despues_na_tel = df["TELEFONO"].isna().sum()

assert df["TELEFONO"].dropna().str.fullmatch(r"\d{4}-\d{4}").all(), "Quedaron valores de TELEFONO fuera del formato NNNN-NNNN"
assert df["TELEFONO_2"].dropna().str.fullmatch(r"\d{4}-\d{4}").all(), "Quedaron valores de TELEFONO_2 fuera del formato NNNN-NNNN"

log.registrar(
    "TELEFONO", f"{antes_vacio_tel} vacíos; {antes_letras_tel} con letras; {antes_multiples_tel} con más de un número; formato inconsistente",
    "Se extraen los números de la celda; el 2do número (si existe) pasa a TELEFONO_2 (derivada); solo se acepta como válido un número de 8 dígitos exactos, formateado NNNN-NNNN; el resto queda NA",
    len(df) - despues_valido_tel,
    "No se inventan dígitos faltantes ni se adivina el número correcto; los casos ambiguos (varios guiones, palabras conectoras como 'Y'/'AL') se resuelven de forma conservadora hacia NA en vez de arriesgar un número incorrecto.",
)
log.registrar(
    "TELEFONO_2", "No existía en la fuente",
    "Variable derivada: guarda el segundo número cuando una celda de TELEFONO original traía más de uno", int(df['TELEFONO_2'].notna().sum()),
    "Evita descartar información real (184 celdas con más de un número en el crudo); se documenta en el Libro de Códigos como derivada.",
)
print(f"TELEFONO: {despues_valido_tel} válidos (formato NNNN-NNNN), {despues_na_tel} NA.")
print(f"TELEFONO_2: {df['TELEFONO_2'].notna().sum()} valores recuperados como segundo número.")


[Fase 3 - Teléfono, duplicados y ensamblaje] TELEFONO: Se extraen los números de la celda; el 2do número (si existe) pasa a TELEFONO_2 (derivada); solo se acepta como válido un número de 8 dígitos exactos, formateado NNNN-NNNN; el resto queda NA -> 1052 registros afectados
[Fase 3 - Teléfono, duplicados y ensamblaje] TELEFONO_2: Variable derivada: guarda el segundo número cuando una celda de TELEFONO original traía más de uno -> 121 registros afectados
TELEFONO: 10815 válidos (formato NNNN-NNNN), 1052 NA.
TELEFONO_2: 121 valores recuperados como segundo número.


## Posibles duplicados parciales

La guía exige buscar duplicados parciales por similitud de cadena (Jaro-Winkler/RapidFuzz) y **documentar la decisión sin eliminar ni fusionar automáticamente**. Se compara `ESTABLECIMIENTO` por pares dentro de cada `MUNICIPIO` (acota la comparación a lo relevante) con RapidFuzz. Para cada par por encima del umbral se aplica una regla de triage transparente y se guarda todo en un CSV aparte para que el grupo lo revise a mano — el dataset principal solo recibe un **flag**, ninguna fila se borra ni se combina aquí.


In [3]:
UMBRAL_SIMILITUD = 90.0

def encontrar_pares_similares(df, umbral=UMBRAL_SIMILITUD):
    pares = []
    for municipio, grupo in df.groupby("MUNICIPIO", dropna=True):
        indices = grupo.index.tolist()
        nombres = grupo["ESTABLECIMIENTO"].fillna("").tolist()
        direcciones = grupo["DIRECCION"].fillna("").tolist()
        jornadas = grupo["JORNADA"].astype(str).tolist()
        planes = grupo["PLAN"].astype(str).tolist()
        n = len(indices)
        if n < 2 or n > 500:
            continue
        for i in range(n):
            for j in range(i + 1, n):
                if nombres[i] == "" or nombres[j] == "" or nombres[i] == nombres[j]:
                    continue
                score = fuzz.token_sort_ratio(nombres[i], nombres[j])
                if score < umbral:
                    continue
                misma_direccion = direcciones[i] != "" and direcciones[i] == direcciones[j]
                misma_jornada_plan = (jornadas[i] == jornadas[j]) and (planes[i] == planes[j])
                if misma_direccion and misma_jornada_plan:
                    nota = "REVISAR: posible duplicado verdadero (nombre, dirección, jornada y plan coinciden)"
                elif not misma_jornada_plan:
                    nota = "NO ES DUPLICADO: mismo colegio con jornada/plan distintos (autorización legítima independiente)"
                else:
                    nota = "REVISAR: nombre muy similar, confirmar manualmente"
                pares.append({
                    "municipio": municipio,
                    "codigo_1": df.loc[indices[i], "CODIGO"], "establecimiento_1": nombres[i],
                    "codigo_2": df.loc[indices[j], "CODIGO"], "establecimiento_2": nombres[j],
                    "similitud": round(score, 1), "nota": nota,
                    "idx_1": indices[i], "idx_2": indices[j],
                })
    return pd.DataFrame(pares)

pares_similares = encontrar_pares_similares(df)
print(f"Pares candidatos a duplicado parcial encontrados: {len(pares_similares)}")
pares_similares["nota"].value_counts()


Pares candidatos a duplicado parcial encontrados: 3293


nota
NO ES DUPLICADO: mismo colegio con jornada/plan distintos (autorización legítima independiente)    2394
REVISAR: nombre muy similar, confirmar manualmente                                                  798
REVISAR: posible duplicado verdadero (nombre, dirección, jornada y plan coinciden)                  101
Name: count, dtype: int64

In [4]:
# Union-find simple: agrupa en un mismo POSIBLE_DUPLICADO_GRUPO a filas
# transitivamente conectadas por al menos un par candidato (ej. A~B y B~C
# quedan en el mismo grupo aunque A~C no se haya comparado directamente).
padre = {}

def encontrar(x):
    padre.setdefault(x, x)
    while padre[x] != x:
        padre[x] = padre[padre[x]]
        x = padre[x]
    return x

def unir(a, b):
    ra, rb = encontrar(a), encontrar(b)
    if ra != rb:
        padre[ra] = rb

for _, fila in pares_similares.iterrows():
    unir(fila["idx_1"], fila["idx_2"])

grupos_por_indice = {idx: encontrar(idx) for idx in padre}
raiz_a_id = {raiz: i for i, raiz in enumerate(sorted(set(grupos_por_indice.values())), start=1)}

df["POSIBLE_DUPLICADO_GRUPO"] = pd.NA
df["NOTA_DUPLICADO"] = pd.NA

for idx, raiz in grupos_por_indice.items():
    df.loc[idx, "POSIBLE_DUPLICADO_GRUPO"] = raiz_a_id[raiz]

notas_por_indice = {}
for _, fila in pares_similares.iterrows():
    for idx in (fila["idx_1"], fila["idx_2"]):
        notas_por_indice.setdefault(idx, []).append(fila["nota"])

for idx, notas in notas_por_indice.items():
    df.loc[idx, "NOTA_DUPLICADO"] = " | ".join(sorted(set(notas)))

n_filas_marcadas = df["POSIBLE_DUPLICADO_GRUPO"].notna().sum()
n_grupos = df["POSIBLE_DUPLICADO_GRUPO"].nunique()
assert len(df) == len(df.loc[df["POSIBLE_DUPLICADO_GRUPO"].isna()]) + n_filas_marcadas

ruta_revision = os.path.join(CARPETA_DATA, "revision_posibles_duplicados.csv")
pares_similares.drop(columns=["idx_1", "idx_2"]).to_csv(ruta_revision, index=False, encoding="utf-8-sig")

log.registrar(
    "(nueva) POSIBLE_DUPLICADO_GRUPO / NOTA_DUPLICADO", f"{len(pares_similares)} pares de nombres similares (RapidFuzz token_sort_ratio >= {UMBRAL_SIMILITUD}) agrupados en {n_grupos} clústeres ({n_filas_marcadas} filas involucradas)",
    "Se agregan 2 columnas derivadas que SOLO marcan/documentan el hallazgo (grupo + nota de triage); ninguna fila se elimina ni se fusiona automáticamente. El detalle de cada par queda en data/revision_posibles_duplicados.csv para revisión manual del grupo",
    n_filas_marcadas,
    "La guía exige no eliminar automáticamente y documentar la decisión de cada caso; un colegio puede tener varios CODIGO legítimos por jornada/plan distintos (la nota ya distingue ese caso de uno realmente sospechoso).",
)
print(f"{n_filas_marcadas} filas marcadas en {n_grupos} grupos de posible duplicado -> {ruta_revision}")
df.loc[df['POSIBLE_DUPLICADO_GRUPO'].notna(), ['CODIGO','ESTABLECIMIENTO','MUNICIPIO','JORNADA','PLAN','POSIBLE_DUPLICADO_GRUPO','NOTA_DUPLICADO']].sort_values('POSIBLE_DUPLICADO_GRUPO').head(10)


[Fase 3 - Teléfono, duplicados y ensamblaje] (nueva) POSIBLE_DUPLICADO_GRUPO / NOTA_DUPLICADO: Se agregan 2 columnas derivadas que SOLO marcan/documentan el hallazgo (grupo + nota de triage); ninguna fila se elimina ni se fusiona automáticamente. El detalle de cada par queda en data/revision_posibles_duplicados.csv para revisión manual del grupo -> 2947 registros afectados
2947 filas marcadas en 745 grupos de posible duplicado -> C:\Users\jplop\Downloads\DATAS\scripts\..\data\revision_posibles_duplicados.csv


,CODIGO,ESTABLECIMIENTO,MUNICIPIO,JORNADA,PLAN,POSIBLE_DUPLICADO_GRUPO,NOTA_DUPLICADO
891,00-02-0060-46,INSTITUTO TÉCNICO PRIVADO AMERICANO,ZONA 2,MATUTINA,DIARIO(REGULAR),1,"REVISAR: posible duplicado verdadero (nombre, dirección, jornada y plan coinciden)"
899,00-02-0068-46,INSTITUTO PRIVADO TÉCNICO AMERICANO,ZONA 2,MATUTINA,DIARIO(REGULAR),1,"REVISAR: posible duplicado verdadero (nombre, dirección, jornada y plan coinciden)"
900,00-02-0069-46,INSTITUTO PRÁCTICO MODERNO,ZONA 2,MATUTINA,DIARIO(REGULAR),2,NO ES DUPLICADO: mismo colegio con jornada/plan distintos (autorización legítima indep...
921,00-02-0115-46,INSTITUTO PRACTICO MODERNO,ZONA 2,DOBLE,FIN DE SEMANA,2,NO ES DUPLICADO: mismo colegio con jornada/plan distintos (autorización legítima indep...
911,00-02-0083-46,INSTITUTO PRACTICO MODERNO,ZONA 2,MATUTINA,DIARIO(REGULAR),2,NO ES DUPLICADO: mismo colegio con jornada/plan distintos (autorización legítima indep...
906,00-02-0078-46,INSTITUTO PRÁCTICO MODERNO,ZONA 2,DOBLE,FIN DE SEMANA,2,NO ES DUPLICADO: mismo colegio con jornada/plan distintos (autorización legítima indep...
886,00-02-0055-46,INSTITUTO PRÁCTICO MODERNO,ZONA 2,MATUTINA,DIARIO(REGULAR),2,NO ES DUPLICADO: mismo colegio con jornada/plan distintos (autorización legítima indep...
889,00-02-0058-46,INSTITUTO PRÁCTICO MODERNO,ZONA 2,MATUTINA,DIARIO(REGULAR),2,NO ES DUPLICADO: mismo colegio con jornada/plan distintos (autorización legítima indep...
934,00-02-0138-46,INSTITUTO PRIVADO MIXTO TECNOLÓGICO EXPERIMENTAL FEDERICO TAYLOR,ZONA 2,MATUTINA,DIARIO(REGULAR),3,NO ES DUPLICADO: mismo colegio con jornada/plan distintos (autorización legítima indep...
881,00-02-0049-46,INSTITUTO PRIVADO MIXTO TECNOLOGICO EXPERIMENTAL VOCACIONAL FEDERICO TAYLOR,ZONA 2,DOBLE,FIN DE SEMANA,3,NO ES DUPLICADO: mismo colegio con jornada/plan distintos (autorización legítima indep...


## Ensamblaje final: tipos de dato definitivos

In [5]:
DOMINIOS = {
    "NIVEL": {"DIVERSIFICADO"},
    "SECTOR": {"OFICIAL", "PRIVADO", "MUNICIPAL", "COOPERATIVA"},
    "AREA": {"URBANA", "RURAL", "SIN ESPECIFICAR"},
    "STATUS": {"ABIERTA", "CERRADA DEFINITIVAMENTE", "CERRADA TEMPORALMENTE", "TEMPORAL NOMBRAMIENTO", "TEMPORAL TITULOS"},
    "MODALIDAD": {"MONOLINGUE", "BILINGUE"},
    "JORNADA": {"DOBLE", "INTERMEDIA", "MATUTINA", "NOCTURNA", "SIN JORNADA", "VESPERTINA"},
    "DEPARTAMENTO": set(catalogo["DEPARTAMENTO"].str.upper().str.strip()),
    "PLAN": {
        "A DISTANCIA", "DIARIO(REGULAR)", "DOMINICAL", "FIN DE SEMANA", "INTERCALADO", "IRREGULAR", "MIXTO",
        "SABATINO", "SEMIPRESENCIAL", "SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)", "SEMIPRESENCIAL (FIN DE SEMANA)",
        "SEMIPRESENCIAL (UN DÍA A LA SEMANA)", "VIRTUAL A DISTANCIA",
    },
}

for col, dominio in DOMINIOS.items():
    valores_reales = set(df[col].dropna().unique())
    inesperados = valores_reales - dominio
    assert not inesperados, f"{col} tiene categorías fuera del dominio esperado tras las 3 fases: {inesperados}"
    df[col] = df[col].astype("category")

print("Tipos de dato finales:")
df.dtypes


Tipos de dato finales:


CODIGO                       object
DISTRITO                     object
DEPARTAMENTO               category
MUNICIPIO                    object
ESTABLECIMIENTO              object
DIRECCION                    object
TELEFONO                     object
SUPERVISOR                   object
DIRECTOR                     object
NIVEL                      category
SECTOR                     category
AREA                       category
STATUS                     category
MODALIDAD                  category
JORNADA                    category
PLAN                       category
DEPARTAMENTAL                object
ARCHIVO_ORIGEN               object
TELEFONO_2                   object
POSIBLE_DUPLICADO_GRUPO      object
NOTA_DUPLICADO               object
dtype: object

## Actividad 7 — Pruebas automáticas de validación del conjunto limpio


In [6]:
resultados_pruebas = []

def prueba(nombre, condicion):
    resultados_pruebas.append((nombre, "PASS" if condicion else "FAIL"))
    assert condicion, f"PRUEBA FALLIDA: {nombre}"

prueba("No existen registros duplicados exactos", not df.duplicated(subset=[c for c in df.columns if c not in ("POSIBLE_DUPLICADO_GRUPO", "NOTA_DUPLICADO")]).any())
prueba("No existen espacios al inicio/final en columnas de texto", not any(
    df[c].dropna().astype(str).str.contains(r"^\s|\s$", regex=True).any()
    for c in ["ESTABLECIMIENTO", "DIRECCION", "SUPERVISOR", "DIRECTOR", "DISTRITO", "DEPARTAMENTAL"]
))
prueba("TELEFONO y TELEFONO_2 tienen formato consistente NNNN-NNNN (o NA)", (
    df["TELEFONO"].dropna().str.fullmatch(r"\d{4}-\d{4}").all()
    and df["TELEFONO_2"].dropna().str.fullmatch(r"\d{4}-\d{4}").all()
))
prueba("DEPARTAMENTO pertenece al catálogo oficial", set(df["DEPARTAMENTO"].dropna().unique()) <= set(catalogo["DEPARTAMENTO"].str.upper().str.strip()))
prueba("Todos los pares (DEPARTAMENTO, MUNICIPIO) pertenecen al catálogo oficial", (
    set(zip(df["DEPARTAMENTO"].astype(str), df["MUNICIPIO"].astype(str)))
    <= set(zip(catalogo["DEPARTAMENTO"].str.upper().str.strip(), catalogo["MUNICIPIO"].str.upper().str.strip()))
))
prueba("Las variables de dominio cerrado tienen tipo 'category'", all(
    str(df[c].dtype) == "category" for c in DOMINIOS
))
prueba("CODIGO cumple el patrón ##-##-####-## y es único", (
    df["CODIGO"].str.fullmatch(r"\d{2}-\d{2}-\d{4}-\d{2}").all() and df["CODIGO"].is_unique
))
prueba("No hay categorías duplicadas por diferencia de escritura (dominio cerrado == catálogo verificado)", all(
    set(df[c].dropna().unique()) == (set(df[c].dropna().unique()) & (DOMINIOS[c] if c != "DEPARTAMENTO" else DOMINIOS["DEPARTAMENTO"]))
    for c in DOMINIOS
))
prueba("No quedan valores inválidos de TELEFONO detectados en el diagnóstico (letras, <8 dígitos sin marcar)", not (
    df["TELEFONO"].dropna().astype(str).str.contains(r"[A-Za-z]", regex=True).any()
))

tabla_pruebas = pd.DataFrame(resultados_pruebas, columns=["prueba", "resultado"])
tabla_pruebas


,prueba,resultado
0,No existen registros duplicados exactos,PASS
1,No existen espacios al inicio/final en columnas de texto,PASS
2,TELEFONO y TELEFONO_2 tienen formato consistente NNNN-NNNN (o NA),PASS
3,DEPARTAMENTO pertenece al catálogo oficial,PASS
4,"Todos los pares (DEPARTAMENTO, MUNICIPIO) pertenecen al catálogo oficial",PASS
5,Las variables de dominio cerrado tienen tipo 'category',PASS
6,CODIGO cumple el patrón ##-##-####-## y es único,PASS
7,No hay categorías duplicadas por diferencia de escritura (dominio cerrado == catálogo ...,PASS
8,"No quedan valores inválidos de TELEFONO detectados en el diagnóstico (letras, <8 dígit...",PASS


## Actividad 8 — Informe de calidad: antes vs. después


In [7]:
df_crudo = pd.read_csv(RUTA_CRUDO, encoding="utf-8-sig", dtype=str, keep_default_na=False)
cols_dato_crudo = [c for c in df_crudo.columns if c != "ARCHIVO_ORIGEN"]
vacias_crudo = df_crudo[cols_dato_crudo].apply(lambda col: col.map(es_vacio)).all(axis=1)
df_crudo_util = df_crudo.loc[~vacias_crudo]

def es_vacio_o_na(v):
    return pd.isna(v) or es_vacio(v)

faltantes_crudo = sum(df_crudo_util[c].map(es_vacio).sum() for c in cols_dato_crudo)
celdas_crudo = len(df_crudo_util) * len(cols_dato_crudo)

cols_dato_limpio = [c for c in df.columns if c not in ("ARCHIVO_ORIGEN", "POSIBLE_DUPLICADO_GRUPO", "NOTA_DUPLICADO", "TELEFONO_2")]
faltantes_limpio = sum(df[c].isna().sum() for c in cols_dato_limpio)
celdas_limpio = len(df) * len(cols_dato_limpio)

log_fase1 = pd.read_csv(os.path.join(CARPETA_INTERIM, "log_fase1.csv"), encoding="utf-8-sig")
log_fase2 = pd.read_csv(os.path.join(CARPETA_INTERIM, "log_fase2.csv"), encoding="utf-8-sig")
log_total = pd.concat([log_fase1, log_fase2, log.a_dataframe()], ignore_index=True)

informe = pd.DataFrame([
    ("Registros", len(df_crudo_util), len(df)),
    ("Variables", len(cols_dato_crudo), len(cols_dato_limpio)),
    ("Valores faltantes", f"{faltantes_crudo} ({100*faltantes_crudo/celdas_crudo:.2f}%)", f"{faltantes_limpio} ({100*faltantes_limpio/celdas_limpio:.2f}%)"),
    ("Variables con NA", sum(df_crudo_util[c].map(es_vacio).any() for c in cols_dato_crudo), sum(df[c].isna().any() for c in cols_dato_limpio)),
    ("Duplicados exactos", int(df_crudo_util.duplicated(subset=cols_dato_crudo).sum()), int(df.duplicated(subset=[c for c in df.columns if c not in ('POSIBLE_DUPLICADO_GRUPO','NOTA_DUPLICADO')]).sum())),
    ("Posibles duplicados", "No evaluado (requiere similitud de cadena)", f"{len(pares_similares)} pares encontrados, {n_filas_marcadas} filas marcadas para revisión (0 fusionados/eliminados automáticamente)"),
    ("Variables con formato inconsistente", "TELEFONO, DISTRITO, ESTABLECIMIENTO (comillas), DIRECCION (abreviaturas)", "0 (todas corregidas o documentadas como texto libre por diseño, ver DISTRITO)"),
    ("Variables con tipo incorrecto", f"{len(DOMINIOS)} categóricas guardadas como texto genérico", "0 (casteadas a category)"),
    ("Categorías inconsistentes", 0, 0),
    ("Errores corregidos", "-", int(log_total['registros_afectados'].sum())),
], columns=["Métrica", "Antes", "Después"])

informe


,Métrica,Antes,Después
0,Registros,11867,11867
1,Variables,17,17
2,Valores faltantes,3826 (1.90%),4037 (2.00%)
3,Variables con NA,6,6
4,Duplicados exactos,0,0
5,Posibles duplicados,No evaluado (requiere similitud de cadena),"3293 pares encontrados, 2947 filas marcadas para revisión (0 fusionados/eliminados aut..."
6,Variables con formato inconsistente,"TELEFONO, DISTRITO, ESTABLECIMIENTO (comillas), DIRECCION (abreviaturas)","0 (todas corregidas o documentadas como texto libre por diseño, ver DISTRITO)"
7,Variables con tipo incorrecto,8 categóricas guardadas como texto genérico,0 (casteadas a category)
8,Categorías inconsistentes,0,0
9,Errores corregidos,-,13475


Dos números que a primera vista podrían parecer un retroceso, pero no lo son:

- **`Valores faltantes` sube de 1.90% a 2.00%.** Esto es correcto y esperado: el crudo solo contaba como faltante una celda vacía o de puros espacios. La limpieza además convierte a `NA` explícito los placeholders disfrazados (`"--"`, `"."`, `"SIN DATO"`, etc. en `DIRECCION`/`DIRECTOR`) y los teléfonos que no se pudieron validar como un número de 8 dígitos reales. El total de faltantes **reales** no cambió — lo que cambió es que ahora quedan **marcados** en vez de escondidos detrás de un placeholder o un número inventado. Ocultar menos problemas es el objetivo de esta actividad, no maquillar el conteo.
- **`Categorías inconsistentes` queda en 0 tanto antes como después.** El diagnóstico (Actividad 3) ya había confirmado que las variables de dominio cerrado llegaron sin variantes de escritura (todas en mayúsculas, sin duplicados tipo "Guatemala"/"GUATEMALA"/"Guatemla"), así que no hubo nada que unificar en esa dimensión — el trabajo real de esta limpieza estuvo en formato (`TELEFONO`, `DISTRITO`, comillas de `ESTABLECIMIENTO`), tipos de dato, y faltantes disfrazados.


## Guardar el conjunto limpio final (Actividad 9) y el registro de transformaciones consolidado (Actividad 6)

In [8]:
COLUMNAS_FINALES = [
    "CODIGO", "DISTRITO", "DEPARTAMENTO", "MUNICIPIO", "ESTABLECIMIENTO", "DIRECCION",
    "TELEFONO", "TELEFONO_2", "SUPERVISOR", "DIRECTOR", "NIVEL", "SECTOR", "AREA", "STATUS",
    "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL", "POSIBLE_DUPLICADO_GRUPO", "NOTA_DUPLICADO",
    "ARCHIVO_ORIGEN",
]
assert set(COLUMNAS_FINALES) == set(df.columns), f"Descuadre de columnas: {set(df.columns).symmetric_difference(COLUMNAS_FINALES)}"
df = df[COLUMNAS_FINALES]

ruta_final = os.path.join(CARPETA_DATA, "establecimientos_diversificado_limpio.csv")
df.to_csv(ruta_final, index=False, encoding="utf-8-sig")

ruta_log_final = os.path.join(CARPETA_DATA, "registro_transformaciones.csv")
log_total.to_csv(ruta_log_final, index=False, encoding="utf-8-sig")

print(f"Conjunto limpio final: {df.shape[0]} filas, {df.shape[1]} columnas -> {ruta_final}")
print(f"Registro de transformaciones consolidado ({len(log_total)} entradas) -> {ruta_log_final}")


Conjunto limpio final: 11867 filas, 21 columnas -> C:\Users\jplop\Downloads\DATAS\scripts\..\data\establecimientos_diversificado_limpio.csv
Registro de transformaciones consolidado (25 entradas) -> C:\Users\jplop\Downloads\DATAS\scripts\..\data\registro_transformaciones.csv
